# Mean-variance optimizer (MVT) for FantasyGo

The "beat the opponent" optimizer of FPL-Optimal-Transfer (`FPL Data/beta estimators/opponent betas/optimization.ipynb`),
applied to the FantasyGo contests of 2023-24.

Choose the team $w$ that maximizes the probability of beating a benchmark score:

$$\max_{w\in\mathbb{W}} \text{Prob}\{w^T\delta > G^{(r')}(\mathbb{W}_o, \delta)\}$$

* $\delta$: the uncertain points of every player, $\delta \sim N(\mu_\delta, \Sigma_\delta)$
* $\mathbb{W}_o$: the opponents' teams; $G_o = w_o^T\delta$ is one team's score and $G^{(r')}$ the score at rank
  `R_PCT` among them (0.5 = beat the median team, 0.8 = finish in the top 20%)
* $\mathbb{W}$: the teams allowed by the FantasyGo rules

**Theorem** (mean-variance reduction): with $Y_w = w^T\delta - G^{(r')}$,
$\mu_{Y_w} = w^T\mu_\delta - \mu_{G}$ and $\sigma^2_{Y_w} = w^T\Sigma_\delta w + \sigma^2_{G} - 2w^T\sigma_{\delta,G}$, and the
solution is $w(\lambda) \in \arg\max_w \; w^T\mu_\delta \mp \lambda\,(w^T\Sigma_\delta w - 2w^T\sigma_{\delta,G})$
($-$ when $\mu_{Y_w}\geq 0$) for some $\lambda \geq 0$. $\mu_G$, $\sigma^2_G$ and $\sigma_{\delta,G}$ do not depend on $w$
and are estimated by Monte Carlo.

**Algorithm**, per gameweek:

1. $\mu_\delta$, $\Sigma_\delta$ from the predictions and their past errors
2. $\mathbb{W}_o$: the opponents' teams
3. Monte Carlo draws of $(\delta, G^{(r')})$
4. Solve for $w(\lambda)$ over a grid of $\lambda$ (mixed-integer quadratic program, SCIP)
5. $\lambda^*$: the $\lambda$ whose team beats $G^{(r')}$ in the most Monte Carlo draws

**Adapted to FantasyGo** (the original picks a 15-man FPL squad with transfers against one head-to-head opponent):

* $w$ is the 11 players with the captain counted 2x and the vice-captain 1.5x: 1 GK, 3-5 DEF, 2-5 MID, 1-3 FWD,
  a budget of 80, at most 3 players per club, unlimited changes each gameweek
* $\mathbb{W}_o$ is the contest field, either simulated (`FIELD = "simulated"`, the default) with the team generator
  of FPL-Optimal-Transfer's `team_modeling - team generator.ipynb` (Algorithm 1), fed by the selection model of
  `beta estimators.ipynb`, or every FantasyGo manager's actual team of the previous gameweek (`FIELD = "previous"`)
* $\mu_\delta$ is `pred` from `models/predictions.csv` (every player with a fixture). $\Sigma_\delta$ is the covariance of
  the errors against the actual points (`total_points - pred`), not xP as in the original: FantasyGo is scored on
  points, and this includes the risk of a player not playing
* players without enough error history get their position's median variance, or, before any player has
  `MIN_HISTORY` gameweeks, the position's variance pooled over every past player-gameweek (the original gave them ~0)
* the decision is restricted to the `SHORTLIST` best-predicted players per position, and $\Sigma_\delta$ is factored
  ($\Sigma = LL^T$) so the quadratic term is a sum of squares, which SCIP solves far faster

In [44]:
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
from pyscipopt import Model, quicksum

ROOT = Path("../..")

preds = pd.read_csv(ROOT / "models/predictions.csv")

all_data = pd.read_csv(
    ROOT / "inputs/rolled_player_data_23-24_all_players_xp_fixed.csv"
)

preds = preds.merge(
    all_data[["fpl_id", "round", "minutes_rolling_3"]],
    on=["fpl_id", "round"],
    how="left",
)

preds["value"] = preds["value"] / 10

preds = preds[preds["minutes_rolling_3"] > 30]  # .sample(50)

fg_picks = pd.read_csv(ROOT / "optimizer/data/fg_manager_picks.csv")
fg_managers = pd.read_csv(ROOT / "optimizer/data/fg_managers.csv")
alpha_by_gw = joblib.load("league_alpha_draws.pkl")  # from beta estimators.ipynb

### Parameters

In [45]:
FIELD = "simulated"  # "simulated" (team generator) or "previous" (last gameweek's actual teams)
SIGMA = "structured"  # "structured" (by position and relationship, sigma.py) or "sample" (per-pair covariance)
SHARED_P = True  # simulated field: one selection vector per position for the whole field, as in the selection
# model's likelihood (True), or one per simulated team (False, the earlier version)
RESULTS = Path(
    f"mvt_results_{FIELD}{'' if SIGMA == 'sample' else '_structured'}"
    f"{'_sharedp' if FIELD == 'simulated' and SHARED_P else ''}.pkl"
)  # cached solutions: delete to recompute

# Team generator (Algorithm 1)
Q_STACK = 0.35  # probability that a simulated manager stacks teammates
BUDGET_LB_QUANTILE = 0.25  # simulated teams spend at least what 75% of the earlier real teams spent (about 90% of the budget);
# the median (97%) is out of reach in GW29, where the generator's teams cost 64 on average
XP_ANCHOR_PCT = 75  # a stack's anchor must be predicted above this percentile

R_PCT = 0.8  # rank to beat in the field: 0.8 = top 20% of the teams
N_MC = 2000  # Monte Carlo draws
LAMBDA_GRID = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
SHORTLIST = 30  # candidates per position, by predicted points
TIME_LIMIT = 30  # seconds per solve
MIN_HISTORY = 3  # gameweeks of prediction errors needed for Sigma
SEED = 0

BUDGET = 80.0
XI_SIZE = 11
FORMATION = {"GK": (1, 1), "DEF": (3, 5), "MID": (2, 5), "FWD": (1, 3)}
MAX_PER_TEAM = 3
CAPTAIN, VICE = 2.0, 1.5

FIRST_GW = (
    6  # the predictions start at 5, so gameweek 6 has one gameweek of errors for Sigma
)
print(
    f"MVT for gameweeks {FIRST_GW}-{preds['round'].max()}, beating the top {1 - R_PCT:.0%} of the field"
)

MVT for gameweeks 6-38, beating the top 20% of the field


### Step 1: $\mu_\delta$ and $\Sigma_\delta$

$\mu_\delta$ is `pred`. $\Sigma_\delta = \mathrm{diag}(\sigma_\delta)\,\rho_\delta\,\mathrm{diag}(\sigma_\delta)$, with $\sigma_\delta$ the sd of each
player's prediction errors in the gameweeks before `gw` (his position's median with fewer than `MIN_HISTORY`).
`SIGMA` picks $\rho_\delta$:

* `"structured"` (default, as in Haugh & Singal's Appendix C): one correlation per (relationship, position pair), with
  same team, opponents, or 0 for unrelated players, pooled over every such pair in the earlier gameweeks (`sigma.py`).
  `preliquisites.ipynb` estimates it and shows that it predicts next week's errors far better than the per-pair version.
* `"sample"`: the per-pair sample correlation of the errors (at least `MIN_HISTORY` shared gameweeks).

Both are made positive semi-definite by clipping the correlation matrix's eigenvalues and rescaling it to a unit diagonal.

In [46]:
import sigma as structured_sigma


def sample_mu_sigma(gw, min_eig_frac=1e-4):
    preds = preds[preds["minutes"] >= 30]
    players = preds[preds["round"] == gw].set_index("fpl_id")
    hist = preds[preds["round"] < gw]
    errors = (hist["total_points"] - hist["pred"]).rename("error")
    pivot = hist.assign(error=errors).pivot_table(
        index="round", columns="fpl_id", values="error"
    )
    cov = pivot.reindex(columns=players.index).cov(min_periods=MIN_HISTORY)

    var = pd.Series(np.diag(cov.to_numpy()), index=players.index)
    position_median = var[var > 0].groupby(players["pos"]).median()
    # early on no player has MIN_HISTORY gameweeks of errors: the position's variance pooled over
    # every past player-gameweek
    pooled = errors.groupby(hist["pos"]).var()
    position_median = position_median.reindex(pooled.index).fillna(pooled)
    var = var.where(var > 0, players["pos"].map(position_median))  # not enough history

    std = np.sqrt(var.to_numpy())
    corr = cov.fillna(0.0).to_numpy() / np.outer(std, std)
    corr = np.clip((corr + corr.T) / 2, -1, 1)
    np.fill_diagonal(corr, 1.0)
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.clip(eigvals, max(min_eig_frac * eigvals.max(), 1e-8), None)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(corr))
    corr = corr / np.outer(d, d)
    sigma = corr * np.outer(std, std)
    return players, players["pred"].to_numpy(), (sigma + sigma.T) / 2


def build_mu_sigma(gw):
    if SIGMA == "structured":
        return structured_sigma.build_mu_sigma(gw, preds)
    return sample_mu_sigma(gw)


### Step 2: the field $\mathbb{W}_o$

Each team is a row of multipliers over the players (2 captain, 1.5 vice, 1 other). Players without a fixture in
`gw` score nothing and are left out.

**Simulated** (Algorithm 1 of the team generator), one team at a time:

1. draw one posterior draw of the selection model (its $\alpha$ for every candidate), sharpen it with the gameweek's
   power $\kappa$ from `beta estimators.ipynb` ($\alpha^\kappa$, rescaled to the same total, so only the shares change)
   and, per position, a selection vector $p \sim \text{Dirichlet}(\alpha)$. With `SHARED_P` (the default) this is done
   once for the whole field, as in the selection model's likelihood, where every manager of a round picks from the same
   $p_t$; otherwise once per simulated team, which averages the field out to the mean shares $\alpha/\sum\alpha$, far
   flatter than a real field (see "Is the simulated field like the real one?")
2. draw a formation from the FantasyGo managers' formations of the earlier gameweeks
3. sample the players of each position by $p$, without replacement
4. with probability `Q_STACK` the manager stacks: if the best selected forward is predicted above the
   `XP_ANCHOR_PCT` percentile, a midfield teammate (sampled by $p$) replaces the weakest selected midfielder; the same
   for the best defender and a goalkeeper of his team
5. keep the team if it costs between `budget_lower_bound(gw)` and the budget (the `BUDGET_LB_QUANTILE` quantile of
   the real FantasyGo teams' cost in the earlier gameweeks) and has at most 3 players per club;
   otherwise start again
6. captain and vice-captain: the two highest predictions (the rule of `selected_info`)

As many teams are simulated as there were entries in the previous gameweek. Unlike the original, duplicate teams
are kept: a real field has them too.

**Previous**: every FantasyGo manager's team of gameweek `gw - 1`, with their captains and vice-captains.

In [47]:
FORMATIONS = [
    (d, m, 10 - d - m) for d in range(3, 6) for m in range(2, 6) if 1 <= 10 - d - m <= 3
]


def formation_frequencies(gw):
    """Share of each formation (DEF, MID, FWD) among the FantasyGo teams before gw."""
    earlier = fg_picks[fg_picks["gameweek"] < gw]
    counts = earlier.groupby(["row", "position"]).size().unstack(fill_value=0)
    shape = counts.reindex(columns=["DEF", "MID", "FWD"], fill_value=0).apply(
        tuple, axis=1
    )
    freq = (
        shape.value_counts().reindex(FORMATIONS, fill_value=0) + 1
    )  # every valid formation possible
    return freq / freq.sum()


def budget_lower_bound(gw):
    """The BUDGET_LB_QUANTILE quantile of the real FantasyGo teams' cost in the gameweeks before gw."""
    earlier = fg_picks[
        (fg_picks["gameweek"] < gw) & (fg_picks["gameweek"] >= preds["round"].min())
    ]
    value = earlier.merge(
        preds[["round", "fpl_id", "value"]],
        left_on=["gameweek", "player_id"],
        right_on=["round", "fpl_id"],
        how="left",
    ).groupby(["gameweek", "row"])["value"]
    cost = value.sum()[value.count() == XI_SIZE]  # teams with every price known
    return float(cost.quantile(BUDGET_LB_QUANTILE))


def sharpen(alpha, power):
    """alpha^power per posterior draw (row), rescaled to the row's total."""
    sharp = np.exp(power * (np.log(alpha) - np.log(alpha).max(axis=1, keepdims=True)))
    return sharp * (alpha.sum(axis=1, keepdims=True) / sharp.sum(axis=1, keepdims=True))


def candidate_pools(gw, players):
    """Per position: the generator's candidates with a prediction this gameweek, and their sharpened alpha draws."""
    pools = {}
    for pos, d in alpha_by_gw[gw].items():
        keep = np.isin(d["fpl_id"], players.index)
        ids = d["fpl_id"][keep]
        pools[pos] = {
            "ids": ids,
            "alpha": sharpen(d["alpha"][:, keep].astype(float), d["power"]),
            "value": players.loc[ids, "value"].to_numpy(),
            "team": players.loc[ids, "team"].to_numpy(),
            "pred": players.loc[ids, "pred"].to_numpy(),
        }
    return pools


def selection_vector(alpha, rng):
    """p ~ Dirichlet(alpha), robust to very small alphas (gamma draws that all underflow to 0)."""
    g = rng.gamma(alpha)
    p = g / g.sum() if g.sum() > 0 else alpha / alpha.sum()
    p = np.maximum(p, 1e-12)
    return p / p.sum()


def stack(selected, p, pool_from, pool_to, anchor_cut, rng, replace_worst=True):
    """Replace a selected player of pool_to with a teammate of the best selected player of pool_from."""
    best = selected["from"][np.argmax(pool_from["pred"][selected["from"]])]
    if pool_from["pred"][best] < anchor_cut:
        return selected["to"]
    mates = np.where(
        (pool_to["team"] == pool_from["team"][best])
        & ~np.isin(np.arange(len(p)), selected["to"])
    )[0]
    if len(mates) == 0:
        return selected["to"]
    mate = rng.choice(mates, p=p[mates] / p[mates].sum())
    out = selected["to"].copy()
    out[np.argmin(pool_to["pred"][out]) if replace_worst else 0] = mate
    return out


def selection_vectors(pools, rng):
    """One posterior draw of the selection model and, per position, p ~ Dirichlet(alpha) under it."""
    s = rng.integers(len(next(iter(pools.values()))["alpha"]))
    return {pos: selection_vector(pools[pos]["alpha"][s], rng) for pos in pools}


def simulate_team(pools, formations, anchor_cut, budget_lb, rng, p=None):
    """One team (fpl ids, multipliers), or None if it breaks the budget or club rules.

    p: the selection vectors shared by the whole field; None draws this team's own."""
    p = selection_vectors(pools, rng) if p is None else p
    form = formations.index[rng.choice(len(formations), p=formations.to_numpy())]
    sizes = {"GK": 1, "DEF": form[0], "MID": form[1], "FWD": form[2]}
    chosen = {
        pos: rng.choice(len(p[pos]), size=n, replace=False, p=p[pos])
        for pos, n in sizes.items()
    }
    if rng.random() < Q_STACK:
        chosen["MID"] = stack(
            {"from": chosen["FWD"], "to": chosen["MID"]},
            p["MID"],
            pools["FWD"],
            pools["MID"],
            anchor_cut,
            rng,
        )
        chosen["GK"] = stack(
            {"from": chosen["DEF"], "to": chosen["GK"]},
            p["GK"],
            pools["DEF"],
            pools["GK"],
            anchor_cut,
            rng,
            replace_worst=False,
        )

    ids = np.concatenate([pools[pos]["ids"][chosen[pos]] for pos in sizes])
    cost = sum(pools[pos]["value"][chosen[pos]].sum() for pos in sizes)
    clubs = np.concatenate([pools[pos]["team"][chosen[pos]] for pos in sizes])
    if (
        not (budget_lb <= cost <= BUDGET)
        or pd.Series(clubs).value_counts().max() > MAX_PER_TEAM
    ):
        return None
    pred = np.concatenate([pools[pos]["pred"][chosen[pos]] for pos in sizes])
    multiplier = np.ones(len(ids))
    order = np.argsort(-pred)
    multiplier[order[0]], multiplier[order[1]] = CAPTAIN, VICE
    return ids, multiplier


def simulate_field(gw, players, rng, n_teams=None, max_tries=200_000, shared_p=None):
    """n_teams simulated teams (default: the previous gameweek's number of entries) and the acceptance rate.

    shared_p: one set of selection vectors for the whole field (default: SHARED_P)."""
    n_teams = n_teams or fg_picks.loc[fg_picks["gameweek"] == gw - 1, "row"].nunique()
    shared_p = SHARED_P if shared_p is None else shared_p
    pools = candidate_pools(gw, players)
    formations = formation_frequencies(gw)
    anchor_cut = np.percentile(players["pred"], XP_ANCHOR_PCT)
    budget_lb = budget_lower_bound(gw)
    p = selection_vectors(pools, rng) if shared_p else None
    teams, tries = [], 0
    while len(teams) < n_teams and tries < max_tries:
        tries += 1
        team = simulate_team(pools, formations, anchor_cut, budget_lb, rng, p)
        if team is not None:
            teams.append(team)
    return teams, len(teams) / tries


def previous_field(gw):
    """The FantasyGo teams of gameweek gw - 1 as (fpl ids, multipliers)."""
    return entries(gw - 1)


def entries(gw):
    """The FantasyGo teams of gameweek gw as (fpl ids, multipliers)."""
    picks = fg_picks[fg_picks["gameweek"] == gw]
    return [
        (
            g["player_id"].to_numpy(),
            np.where(g["is_captain"], CAPTAIN, np.where(g["is_vice"], VICE, 1.0)),
        )
        for _, g in picks.groupby("row")
    ]


def field_matrix(gw, players, rng):
    """W_o: one row of multipliers over `players` per team of the field."""
    teams = (
        simulate_field(gw, players, rng)[0]
        if FIELD == "simulated"
        else previous_field(gw)
    )
    position = {e: i for i, e in enumerate(players.index)}
    W_o = np.zeros((len(teams), len(players)))
    for k, (ids, multiplier) in enumerate(teams):
        for e, m in zip(ids, multiplier):
            if e in position:
                W_o[k, position[e]] += m
    return W_o

#### Is the simulated field like the real one?

For every gameweek, the simulated field next to the real contest of that gameweek and the previous gameweek's
teams (the other `FIELD`), all scored with the actual points:

* `cost`: mean team cost
* `share_corr`: rank correlation of how often each player is picked, against the real field
* `cut_points`: the points needed for the top `1 - R_PCT` of the field, the $G^{(r')}$ the optimizer has to beat

In [48]:
# def field_summary(teams, gw):
#     actual = preds[preds["round"] == gw].set_index("fpl_id")
#     points = [
#         (actual["total_points"].reindex(ids).fillna(0).to_numpy() * m).sum()
#         for ids, m in teams
#     ]
#     cost = [actual["value"].reindex(ids).sum() for ids, _ in teams]
#     picks = pd.Series(np.concatenate([ids for ids, _ in teams])).value_counts()
#     return {
#         "cost": np.mean(cost),
#         "cut_points": np.quantile(points, R_PCT),
#         "mean_points": np.mean(points),
#     }, picks


# rows = []
# for gw in range(FIRST_GW, preds["round"].max() + 1):
#     rng = np.random.default_rng(SEED + gw)
#     players = preds[preds["round"] == gw].set_index("fpl_id")
#     real, real_picks = field_summary(entries(gw), gw)
#     sim_teams, accepted = simulate_field(gw, players, rng)
#     for name, teams in [("simulated", sim_teams), ("previous", previous_field(gw))]:
#         summary, picks = field_summary(teams, gw)
#         both = pd.concat([picks, real_picks], axis=1).fillna(0)
#         rows.append(
#             {
#                 "round": gw,
#                 "field": name,
#                 **summary,
#                 "share_corr": both.corr(method="spearman").iloc[0, 1],
#                 "accept_rate": accepted if name == "simulated" else np.nan,
#             }
#         )
#     rows.append({"round": gw, "field": "real", **real})
# fields = pd.DataFrame(rows)
# fields.groupby("field")[
#     ["cost", "cut_points", "mean_points", "share_corr", "accept_rate"]
# ].mean().round(2)

### Step 3: Monte Carlo $(\delta, G^{(r')})$

$G^{(r')}$ is the score of the team at rank `R_PCT` of the field in each draw.

In [49]:
def monte_carlo(mu, sigma, W_o, rng):
    delta = rng.multivariate_normal(mu, sigma, size=N_MC, method="eigh")
    k = min(
        max(int(np.ceil((1 - R_PCT) * len(W_o))), 1), len(W_o)
    )  # r' counted from the top
    G = np.sort(delta @ W_o.T, axis=1)[:, -k]
    cov_delta_G = ((delta - mu) * (G - G.mean())[:, None]).mean(axis=0)
    return delta, G, cov_delta_G

### Step 4: $w(\lambda)$, a mixed-integer quadratic program

Variables per candidate: picked, captain and vice-captain (binary), and its multiplier
$w_i = \text{picked} + \text{captain} + 0.5\,\text{vice}$. With $\Sigma = LL^T$ and $y = L^Tw$, the variance is
$\sum_k y_k^2$. SCIP takes a linear objective, so the mean-variance objective is the variable $z$ with
$z \leq w^T\mu - s\lambda(\sum_k y_k^2 - 2w^T\sigma_{\delta,G})$. For $s = +1$ ($\mu_{Y_w} \geq 0$) this is convex
and solves quickly; for $s = -1$ it is not, and the solve stops at `TIME_LIMIT` with its best team.

In [50]:
def shortlist(players):
    best = (
        players.reset_index()
        .groupby("pos", group_keys=False)
        .apply(lambda g: g.nlargest(SHORTLIST, "pred"))
    )
    return best.index.to_numpy()  # positions in `players`


def solve_for_lambda(cands, mu, sigma, cov_delta_G, lam, sense=1):
    n = len(cands)
    m = Model()
    m.hideOutput()
    m.setParam("limits/time", TIME_LIMIT)
    pick = [m.addVar(vtype="B") for _ in range(n)]
    captain = [m.addVar(vtype="B") for _ in range(n)]
    vice = [m.addVar(vtype="B") for _ in range(n)]
    w = [m.addVar(lb=0, ub=CAPTAIN) for _ in range(n)]
    for i in range(n):
        m.addCons(w[i] == pick[i] + (CAPTAIN - 1) * captain[i] + (VICE - 1) * vice[i])
        m.addCons(captain[i] + vice[i] <= pick[i])
    m.addCons(quicksum(pick) == XI_SIZE)
    m.addCons(quicksum(captain) == 1)
    m.addCons(quicksum(vice) == 1)
    m.addCons(quicksum(cands["value"].iat[i] * pick[i] for i in range(n)) <= BUDGET)
    for pos, (low, high) in FORMATION.items():
        n_pos = quicksum(pick[i] for i in np.where(cands["pos"].to_numpy() == pos)[0])
        m.addCons(n_pos >= low)
        m.addCons(n_pos <= high)
    for team in cands["team"].unique():
        m.addCons(
            quicksum(pick[i] for i in np.where(cands["team"].to_numpy() == team)[0])
            <= MAX_PER_TEAM
        )

    eigvals, eigvecs = np.linalg.eigh(sigma)
    keep = eigvals > 1e-9 * eigvals.max()
    L = eigvecs[:, keep] * np.sqrt(eigvals[keep])
    y = [m.addVar(lb=-1e4, ub=1e4) for _ in range(L.shape[1])]
    for k in range(L.shape[1]):
        m.addCons(y[k] == quicksum(L[i, k] * w[i] for i in range(n)))
    mean = quicksum(mu[i] * w[i] for i in range(n))
    variance = quicksum(v * v for v in y) - 2 * quicksum(
        cov_delta_G[i] * w[i] for i in range(n)
    )
    z = m.addVar(lb=-1e6, ub=1e6)
    m.addCons(z <= mean - sense * lam * variance)
    m.setObjective(z, "maximize")
    m.optimize()
    if m.getNSols() == 0:
        return None, m.getStatus()
    sol = m.getBestSol()
    team = cands.assign(
        w=[sol[v] for v in w],
        captain=[sol[v] > 0.5 for v in captain],
        vice=[sol[v] > 0.5 for v in vice],
    )
    return team[[sol[v] > 0.5 for v in pick]], m.getStatus()

### Step 5: $\lambda^*$, and the whole gameweek as a function

$\widehat{\text{Prob}}(Y_w > 0)$ is the share of the Monte Carlo draws where the team beats $G^{(r')}$. The sign
$s$ comes from the best-mean team ($\lambda = 0$).

In [51]:
def run_mvt(gw, verbose=True):
    rng = np.random.default_rng(SEED + gw)
    players, mu, sigma = build_mu_sigma(gw)
    W_o = field_matrix(gw, players, rng)
    delta, G, cov_delta_G = monte_carlo(mu, sigma, W_o, rng)

    idx = shortlist(players)
    cands = players.reset_index().iloc[idx].reset_index(drop=True)
    mu_c, sigma_c, cov_c = mu[idx], sigma[np.ix_(idx, idx)], cov_delta_G[idx]

    frontier, teams, sense = [], {}, 1
    for lam in LAMBDA_GRID:
        team, status = solve_for_lambda(cands, mu_c, sigma_c, cov_c, lam, sense)
        if team is None:
            continue
        w = np.zeros(len(cands))
        w[team.index] = team["w"]
        if lam == 0:
            sense = 1 if mu_c @ w - G.mean() >= 0 else -1
        Y = delta[:, idx] @ w - G
        frontier.append(
            {
                "lambda": lam,
                "status": status,
                "prob_beat": (Y > 0).mean(),
                "mean": mu_c @ w,
                "sd": np.sqrt(w @ sigma_c @ w),
            }
        )
        teams[lam] = team
    frontier = pd.DataFrame(frontier)
    best = frontier.loc[frontier["prob_beat"].idxmax(), "lambda"]
    if verbose:
        print(
            f"GW{gw}: field of {len(W_o)}, G mean {G.mean():.1f} sd {G.std():.1f}, "
            f"lambda* = {best}, Prob(beat) {frontier['prob_beat'].max():.1%}"
        )
    return {
        "gw": gw,
        "frontier": frontier,
        "teams": teams,
        "lambda_star": best,
        "G_mean": G.mean(),
        "G_sd": G.std(),
    }

### One gameweek

In [52]:
preds[preds["round"] == 30]

,pos,round,fpl_id,value,name,team,total_points,selected_at,model,pred,xP,in_selection,minutes,played,minutes_rolling_3
2123,GK,30,49,5.3,Emiliano Martínez Romero,Aston Villa,10,25,rf,2.480615,4.5,False,90.0,True,90.000000
2126,GK,30,77,4.7,Norberto Murara Neto,Bournemouth,2,25,rf,3.196270,3.5,False,90.0,True,60.000000
2131,GK,30,101,4.6,Mark Flekken,Brentford,3,25,rf,2.199589,2.2,False,90.0,True,90.000000
2147,GK,30,230,4.4,Sam Johnstone,Crystal Palace,0,25,rf,1.223029,0.0,False,0.0,False,60.000000
2150,GK,30,263,4.6,Jordan Pickford,Everton,2,25,rf,2.680173,1.7,False,90.0,True,60.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25164,FWD,30,523,5.7,Michail Antonio,West Ham,6,25,lgbm,2.947907,4.0,False,68.0,True,40.000000
25186,FWD,30,691,5.7,Norberto Bercique Gomes Betuncal,Everton,8,25,lgbm,3.540527,4.7,False,11.0,True,44.333333
25187,FWD,30,704,4.5,Nathan Fraser,Wolves,1,25,lgbm,0.623488,0.8,False,6.0,True,40.333333
25188,FWD,30,714,5.0,Divock Origi,Nott'm Forest,1,25,lgbm,2.028417,1.8,False,59.0,True,66.000000


In [53]:
rng = np.random.default_rng(SEED + 30)

players, mu, sigma = build_mu_sigma(30)
W_o = field_matrix(30, players, rng)
W_o

KeyboardInterrupt: 

In [54]:
example = run_mvt(30)
example["frontier"].round(3)

GW30: field of 228, G mean 68.6 sd 14.0, lambda* = 0.05, Prob(beat) 80.9%


,lambda,status,prob_beat,mean,sd
0,0.000,optimal,0.798,77.995,19.972
1,0.005,optimal,0.805,77.995,20.145
2,0.010,optimal,0.805,77.995,20.145
3,0.020,optimal,0.805,77.995,20.145
4,0.050,optimal,0.809,77.377,20.670
5,0.100,timelimit,0.776,76.256,20.204
6,0.200,timelimit,0.746,75.591,18.328
7,0.500,timelimit,0.458,67.259,17.326


In [55]:
example["teams"][example["lambda_star"]][
    ["pos", "name", "team", "value", "pred", "captain", "vice"]
]

,pos,name,team,value,pred,captain,vice
0,DEF,Benjamin White,Arsenal,5.7,5.867908,False,False
1,DEF,Cristian Romero,Spurs,5.1,5.827739,False,False
2,DEF,Malo Gusto,Chelsea,4.2,4.856106,False,False
30,FWD,Rodrigo Muniz Carvalho,Fulham,4.6,7.243659,False,False
31,FWD,Ollie Watkins,Aston Villa,9.0,6.008747,False,True
32,FWD,Erling Haaland,Man City,14.3,5.610996,False,False
54,GK,Bernd Leno,Fulham,4.8,5.577265,False,False
72,MID,Son Heung-min,Spurs,10.1,7.276041,True,False
73,MID,Cole Palmer,Chelsea,5.8,6.732080,False,False
74,MID,Phil Foden,Man City,8.1,6.303892,False,False


In [56]:
example["teams"][example["lambda_star"]][
    ["pos", "name", "team", "value", "pred", "captain", "vice"]
]["pred"].sum()

np.float64(67.09667808373635)

### Every gameweek

Solutions are cached in `mvt_results.pkl` (delete it, or change a parameter and delete it, to recompute).

In [ ]:
import pickle

if RESULTS.exists():
    results = pickle.loads(RESULTS.read_bytes())
else:
    results = {gw: run_mvt(gw) for gw in range(FIRST_GW, preds["round"].max() + 1)}
    RESULTS.write_bytes(pickle.dumps(results))

### How did the teams actually do?

Each team scored on the actual points (captain x2, vice x1.5) and ranked in the gameweek's FantasyGo contest:
rank = 1 + entries with more points; `prize` is what the entry at that rank won. The mean-variance team
($\lambda^*$) is compared with the best-mean team ($\lambda = 0$, the base optimizer on the same shortlist).

In [ ]:
def team_points(team, gw):
    actual = preds[preds["round"] == gw].set_index("fpl_id")["total_points"]
    return float((team["fpl_id"].map(actual).fillna(0) * team["w"]).sum())


def contest_rank(points, gw):
    field = fg_managers[fg_managers["gameweek"] == gw].sort_values(
        "points", ascending=False
    )
    rank = int((field["points"] > points).sum()) + 1
    prize = field["prize"].iloc[rank - 1] if rank <= len(field) else 0.0
    return rank, len(field), prize


rows = []
for gw, res in results.items():
    row = {"round": gw, "lambda_star": res["lambda_star"], "G_mean": res["G_mean"]}
    for label, lam in [("mvt", res["lambda_star"]), ("base", 0.0)]:
        team = res["teams"][lam]
        f = res["frontier"].set_index("lambda").loc[lam]
        points = team_points(team, gw)
        rank, entries, prize = contest_rank(points, gw)
        row |= {
            f"{label}_prob": f["prob_beat"],
            f"{label}_mean": f["mean"],
            f"{label}_sd": f["sd"],
            f"{label}_points": points,
            f"{label}_rank": rank,
            f"{label}_prize": prize,
        }
    row["entries"] = entries
    row["target_rank"] = int(np.ceil((1 - R_PCT) * entries))
    rows.append(row)
evaluation = pd.DataFrame(rows).set_index("round").round(2)

for label in ["mvt", "base"]:
    hit = (evaluation[f"{label}_rank"] <= evaluation["target_rank"]).sum()
    print(
        f"{label:4s}: {evaluation[f'{label}_points'].sum():.0f} points, median rank top "
        f"{(evaluation[f'{label}_rank'] / evaluation['entries']).median():.0%}, in the top {1 - R_PCT:.0%} in "
        f"{hit} of {len(evaluation)} gameweeks (predicted {evaluation[f'{label}_prob'].mean():.0%}), "
        f"prizes {evaluation[f'{label}_prize'].sum():,.2f}"
    )
evaluation

mvt : 1810 points, median rank top 59%, in the top 20% in 3 of 33 gameweeks (predicted 82%), prizes 251.01
base: 1846 points, median rank top 63%, in the top 20% in 3 of 33 gameweeks (predicted 80%), prizes 1,303.81


,lambda_star,G_mean,mvt_prob,mvt_mean,mvt_sd,mvt_points,mvt_rank,mvt_prize,base_prob,base_mean,base_sd,base_points,base_rank,base_prize,entries,target_rank
round,,,,,,,,,,,,,,,,
6,0.05,74.39,0.96,82.62,7.75,64.0,160,0.00,0.94,82.74,7.77,60.0,182,0.00,270,54
7,0.20,68.03,0.91,75.59,7.86,65.0,100,0.00,0.89,76.49,7.86,57.5,143,0.00,276,56
8,0.00,77.36,0.65,81.81,21.26,54.0,146,0.00,0.65,81.81,21.26,54.0,146,0.00,269,54
9,0.01,73.95,0.62,76.15,19.69,86.0,140,0.00,0.60,76.39,21.02,87.0,135,0.00,256,52
10,0.00,72.39,0.73,81.15,21.85,51.0,226,0.00,0.72,81.19,21.26,53.5,216,0.00,277,56
11,0.05,79.59,0.83,90.58,19.74,17.5,311,0.00,0.79,91.68,21.01,28.0,245,0.00,313,63
12,0.01,82.97,0.72,91.24,23.12,63.0,158,0.00,0.70,91.55,24.29,52.0,238,0.00,342,69
13,0.02,78.56,0.77,87.17,19.77,40.5,182,0.00,0.73,88.14,20.08,35.5,239,0.00,286,58
14,0.02,79.90,0.79,88.02,20.56,54.5,175,0.00,0.77,88.36,22.42,50.5,211,0.00,299,60


### Simulated field against the previous gameweek's teams

Both runs side by side, when both caches exist (run the notebook once with each `FIELD`).

In [ ]:
def summarize(path):
    res = pickle.loads(Path(path).read_bytes())
    out = []
    for gw, r in res.items():
        team = r["teams"][r["lambda_star"]]
        points = team_points(team, gw)
        rank, entries, prize = contest_rank(points, gw)
        out.append(
            {
                "round": gw,
                "lambda_star": r["lambda_star"],
                "G_mean": r["G_mean"],
                "points": points,
                "rank": rank,
                "top": rank <= np.ceil((1 - R_PCT) * entries),
                "prize": prize,
            }
        )
    return pd.DataFrame(out).set_index("round")


runs = {
    f: summarize(f"mvt_results_{f}.pkl")
    for f in ["simulated", "previous"]
    if Path(f"mvt_results_{f}.pkl").exists()
}
pd.DataFrame(
    {
        f: {
            "points": r["points"].sum(),
            "median_rank": r["rank"].median(),
            "top_20pct_weeks": r["top"].sum(),
            "prizes": r["prize"].sum(),
            "mean_G": r["G_mean"].mean(),
        }
        for f, r in runs.items()
    }
).round(1)

,simulated,previous
points,1892.0,1766.5
median_rank,129.0,182.0
top_20pct_weeks,5.0,2.0
prizes,148.1,25.0
mean_G,75.1,69.1


### Caveats

* $\mu_\delta$ over-predicts the top players (see `optimizer.ipynb`), so $\widehat{\text{Prob}}$ is optimistic: both
  teams and the field are scored with the same biased $\mu_\delta$, but our team is chosen for high $\mu_\delta$.
* $\Sigma_\delta$ rests on 1 to 33 gameweeks of errors (pooled per position for the first few), so its correlations are noisy; the eigenvalue floor keeps
  it usable but does not make it accurate.
* The simulated field rests on the selection model and on the captain rule (two highest predictions); the
  previous-gameweek field ignores that managers can change all 11 players each gameweek.
* Only the `SHORTLIST` best-predicted players per position are candidates.